# Técnicas de Diseño y Validación de Modelos · Universidad Blas Pascal
## Módulo 3 · Overfitting, Underfitting y Diagnóstico
## Laboratorio: Curvas de validación y de aprendizaje

<div style="display:flex; justify-content:flex-start; margin: 8px 0 18px 0;">
  <a href="https://colab.research.google.com/github/GabrielaOjcius/Laboratorios-Tecnicas-Diseno-Validacion-Modelos/blob/main/Laboratorios%20opcionales/M3_Notebook.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/>
  </a>
  <span style="font-size: 0.98em;">En Colab, hac&eacute; una copia en tu Drive antes de editar.</span>
</div>


---

### Objetivo
Construir curvas de validación (error vs. hiperparámetro de complejidad) y curvas de aprendizaje (error vs. tamaño de entrenamiento) para diagnosticar el régimen de generalización de un modelo. El experimento compara un árbol de decisión (alta varianza esperada) con un Random Forest (varianza reducida por bagging).

### Alineación con el material teórico
Este notebook corresponde a la **Actividad Optativa** del documento `Módulo 3 · Overfitting, Underfitting y Diagnóstico`. Prepara la **Sección A** de la Parte 2 de la Evaluación Integradora.

### Instrucciones
1. Ejecutar en orden (Entorno de ejecución → Ejecutar todo).
2. Completar las celdas de respuesta marcadas con `# ✏️ RESPONDA AQUÍ`.
3. No modificar el código de setup sin justificación.

In [ ]:
# ─── B.1. Entorno y dataset de clasificación ─────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     validation_curve, learning_curve)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Dataset: moderadamente complejo, bien definido
X, y = make_classification(
    n_samples=800, n_features=15,
    n_informative=8, n_redundant=4, n_repeated=0,
    class_sep=1.0, random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Train:', X_tr.shape, '  Test:', X_te.shape)
print('Proporción positivos train: %.3f  test: %.3f' % (y_tr.mean(), y_te.mean()))

In [ ]:
# ─── B.2. Curva de validación — Árbol de decisión ────────────────────────────
depths = np.arange(1, 21)

tr_arbol, va_arbol = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X_tr, y_tr,
    param_name='max_depth', param_range=depths,
    cv=cv, scoring='accuracy', n_jobs=-1
)

acc_tr = tr_arbol.mean(axis=1)
acc_va = va_arbol.mean(axis=1)
gap    = acc_tr - acc_va

opt_depth = depths[np.argmax(acc_va)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel izquierdo: curvas de validación
axes[0].plot(depths, acc_tr, '-o', color='steelblue', lw=2, label='Train')
axes[0].plot(depths, acc_va, '-o', color='#8B1A2F', lw=2, label='Validación')
axes[0].fill_between(depths, acc_va, acc_tr, alpha=0.15, color='#8B1A2F', label='Gap')
axes[0].axvline(opt_depth, ls='--', color='gray', label=f'Óptimo (depth={opt_depth})')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Curva de validación — Árbol de decisión')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Panel derecho: gap
axes[1].plot(depths, gap, '-o', color='#8B1A2F', lw=2)
axes[1].axhline(0, ls='--', color='gray')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Gap (Train − Val)')
axes[1].set_title('Gap de generalización en función de la complejidad')
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f'Mejor max_depth: {opt_depth}  |  accuracy val: {acc_va.max():.3f}')
print(f'Gap en max_depth={opt_depth}: {gap[np.argmax(acc_va)]:.3f}')
print(f'Gap máximo (depth=20): {gap[-1]:.3f}')

In [ ]:
# ─── B.3. Curva de validación — Random Forest (comparación) ──────────────────
tr_rf, va_rf = validation_curve(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_tr, y_tr,
    param_name='max_depth', param_range=depths,
    cv=cv, scoring='accuracy', n_jobs=-1
)

acc_tr_rf = tr_rf.mean(axis=1)
acc_va_rf = va_rf.mean(axis=1)
gap_rf    = acc_tr_rf - acc_va_rf

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
fig.suptitle('Árbol simple vs. Random Forest (100 árboles) — Curvas de validación', fontweight='bold')

for ax, (tr, va, gp), titulo in zip(
    axes,
    [(acc_tr, acc_va, gap), (acc_tr_rf, acc_va_rf, gap_rf)],
    ['Árbol de decisión simple', 'Random Forest (100 árboles)']
):
    ax.plot(depths, tr, '-o', color='steelblue', lw=2, label='Train')
    ax.plot(depths, va, '-o', color='#8B1A2F', lw=2, label='Validación')
    ax.fill_between(depths, va, tr, alpha=0.12, color='#8B1A2F')
    ax.set_title(titulo); ax.set_xlabel('max_depth'); ax.legend(); ax.grid(alpha=0.3)

axes[0].set_ylabel('Accuracy')
plt.tight_layout(); plt.show()

print(f'Gap máximo — Árbol simple: {gap.max():.3f}')
print(f'Gap máximo — Random Forest: {gap_rf.max():.3f}')
print(f'Mejor accuracy val — Árbol: {acc_va.max():.3f}  (depth={opt_depth})')
print(f'Mejor accuracy val — RF:    {acc_va_rf.max():.3f}  (depth={depths[np.argmax(acc_va_rf)]})')

In [ ]:
# ─── B.4. Curva de aprendizaje — árbol con mejor max_depth ───────────────────
train_sizes = np.linspace(50, 600, 12, dtype=int)

ts, tr_lc, va_lc = learning_curve(
    DecisionTreeClassifier(max_depth=opt_depth, random_state=42),
    X_tr, y_tr,
    train_sizes=train_sizes,
    cv=cv, scoring='accuracy', n_jobs=-1, random_state=42
)

tr_mean, tr_std = tr_lc.mean(1), tr_lc.std(1)
va_mean, va_std = va_lc.mean(1), va_lc.std(1)

plt.figure(figsize=(8, 5))
plt.fill_between(ts, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color='steelblue')
plt.fill_between(ts, va_mean - va_std, va_mean + va_std, alpha=0.15, color='#8B1A2F')
plt.plot(ts, tr_mean, '-o', color='steelblue', lw=2, label='Train')
plt.plot(ts, va_mean, '-o', color='#8B1A2F', lw=2, label='Validación')
plt.xlabel('Tamaño del conjunto de entrenamiento')
plt.ylabel('Accuracy')
plt.title(f'Curva de aprendizaje — Árbol (max_depth={opt_depth})')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ¿Se aplana la curva de validación?
delta_val = va_mean[-1] - va_mean[-3]  # mejora en los últimos 3 puntos
print(f'Mejora de validación en los últimos 200 ejemplos: {delta_val:.4f}')
print('Diagnóstico: ' + ('la curva sigue mejorando → más datos pueden ayudar' if delta_val > 0.005
                         else 'la curva se aplana → más datos no ayudan significativamente'))

In [ ]:
# ─── B.5. Efecto del ruido irreducible: class_sep = 0.5 ──────────────────────
X2, y2 = make_classification(
    n_samples=800, n_features=15,
    n_informative=8, n_redundant=4,
    class_sep=0.5, random_state=42   # clases más solapadas
)
X2_tr, _, y2_tr, _ = train_test_split(X2, y2, test_size=0.25, stratify=y2, random_state=42)

tr2, va2 = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X2_tr, y2_tr,
    param_name='max_depth', param_range=depths,
    cv=cv, scoring='accuracy', n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(depths, acc_va,          '-o', color='steelblue',  lw=2, label='Val — class_sep=1.0')
plt.plot(depths, va2.mean(axis=1),'-o', color='#8B1A2F',    lw=2, label='Val — class_sep=0.5')
plt.axhline(y_tr.mean(), ls=':', color='steelblue', lw=1,  label='Baseline (sep=1.0)')
plt.axhline(y2_tr.mean(), ls=':',color='#8B1A2F',   lw=1,  label='Baseline (sep=0.5)')
plt.xlabel('max_depth'); plt.ylabel('Accuracy de validación')
plt.title('Efecto de la separabilidad de clases sobre la curva de validación')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Accuracy val máximo — class_sep=1.0: {acc_va.max():.3f}')
print(f'Accuracy val máximo — class_sep=0.5: {va2.mean(axis=1).max():.3f}')
print(f'Caída: {acc_va.max() - va2.mean(axis=1).max():.3f} puntos')

---
## Preguntas de análisis
Responder en las celdas markdown siguientes. Incluir referencias a los valores numéricos de las secciones anteriores.

### Pregunta 1
¿A partir de qué `max_depth` el árbol simple empieza a sobreajustar de manera notoria? Identificar el punto en el gráfico de B.2 y cuantificar el gap en ese punto.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 2
Comparar el gap máximo del árbol simple con el del Random Forest. ¿Por qué el bosque es más resistente al overfitting? Conectar con el concepto de bagging y la reducción de varianza por promediado.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 3
La curva de aprendizaje del árbol (B.4): ¿se aplana al aumentar los datos o sigue mejorando? ¿Qué implica esto para la estrategia de mitigación del gap?

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 4
Si el gap es grande y la curva de aprendizaje muestra que la validación sigue mejorando con más datos, ¿qué estrategia priorizaría: conseguir más datos o regularizar el modelo? Justificar en términos de costo y efectividad.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 5
Al reducir `class_sep` de 1.0 a 0.5, ¿cómo cambia el accuracy máximo alcanzable en validación? Conectar con el concepto de ruido irreducible y explicar por qué ninguna mejora del modelo puede recuperar ese accuracy.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`